# 02 — 모델 아키텍처: 인코더 → 재귀 월드모델 → 디코더

세 가지 모듈로 구성됩니다 (`wm_toy.py` 참고):

### 1) `CamEncoder` — 멀티카메라 퓨전
- 카메라마다 **같은 CNN 가중치를 공유**해서 각 카메라 이미지를 임베딩으로 압축합니다.
  (가중치를 공유하는 이유: "차선이 어떻게 생겼는지"를 카메라마다 따로 배울 필요가 없고,
  실제 서라운드 스택도 보통 카메라 간 가중치를 공유하거나 공유에 가까운 구조를 씁니다.)
- 3개 카메라 임베딩을 이어붙여(concat) 하나의 관측 임베딩(`OBS_EMBED_DIM`)을 만듭니다 —
  이게 "멀티카메라 퓨전"입니다.

### 2) 재귀 월드모델 코어 (`WorldModel.gru` + `embed_head`)
- `GRUCell`이 "지금까지 본 것 + 지금 행동"을 누적한 은닉 상태 `h_t`를 유지합니다 —
  이게 "시계열을 다룬다"는 부분입니다.
- **posterior step** (학습 시, 실제 관측이 있을 때): `h_t = GRU(concat(obs_embed_t, action_t), h_{t-1})`
- **prior / imagination step** (관측 없이 "상상"할 때): 먼저 `embed_head(h_{t-1})`로
  "이번에 뭘 보게 될지"를 스스로 예측한 뒤, 그 예측 임베딩을 진짜 관측 대신 넣어서 `h_t`를
  굴립니다. 이게 월드모델의 핵심 트릭이고, Dreamer류의 RSSM(Recurrent State-Space Model)이나
  2015년 World Models 논문의 MDN-RNN과 같은 계열의 아이디어입니다 (다만 여긴 확률적 잠재변수
  없이 결정론적으로 단순화했습니다).

### 3) `CamDecoder` — 다시 이미지로
- 은닉 상태 하나에서 3개 카메라 이미지를 동시에 복원합니다. 학습 중에는 "내가 본 걸 잘
  재구성하는지"를 확인하는 용도이고, 롤아웃 노트북에서는 "상상한 미래가 어떻게 생겼는지"를
  눈으로 보는 용도로 씁니다.


In [1]:
import torch
import wm_toy as W

model = W.WorldModel()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("device:", device)

train_ds = W.WMSequenceDataset(num_episodes=4, T=40, base_seed=1000)
frames, actions, states = train_ds[0]
frames = frames.unsqueeze(0).to(device)   # add batch dim -> (1, T, cam, 3, H, W)
actions = actions.unsqueeze(0).to(device)

h = model.init_hidden(batch_size=1, device=device)
h, recon, obs_embed = model.posterior_step(h, frames[:, 0], actions[:, 0])
print("hidden state shape:", h.shape)
print("obs_embed shape:   ", obs_embed.shape)
print("reconstruction shape (untrained, so it'll look like noise):", recon.shape)


device: cpu
hidden state shape: torch.Size([1, 128])
obs_embed shape:    torch.Size([1, 96])
reconstruction shape (untrained, so it'll look like noise): torch.Size([1, 3, 3, 64, 64])


In [2]:
# imagination(prior) step도 동작하는지 확인 — 아직 학습 전이라 의미있는 예측은 아니고, shape/배선만 확인
h2, recon2, pred_embed = model.prior_step(h, actions[:, 1])
print("prior step ok. predicted embed shape:", pred_embed.shape, "| recon shape:", recon2.shape)


prior step ok. predicted embed shape: torch.Size([1, 96]) | recon shape: torch.Size([1, 3, 3, 64, 64])


두 스텝(`posterior_step`, `prior_step`)이 **같은 GRUCell과 같은 디코더를 공유**한다는 점이
중요합니다. 학습은 항상 `posterior_step`(teacher forcing)으로 하지만, 모델이 배운 `embed_head`가
있어야 나중에 관측 없이 `prior_step`만으로 여러 스텝을 굴릴 수 있습니다 — 이게 03번에서
학습하고, 04번에서 실제로 "상상"시켜볼 부분입니다.
